# Experiment 2: Known-Structure Recovery

**Purpose:** Verify that SuStaIn can detect genuine subtypes when they exist, validating that failure in null data reflects absence of structure rather than methodological limitation.

**Design:** Generate synthetic data with 3 pre-defined subtypes (Musculoskeletal-dominant, Autonomic-dominant, Neurological-dominant) with ground-truth Cohen's d > 1.0 for domain-defining symptoms. Run SuStaIn and assess recovery via ARI.

**Success criterion:** ARI > 0.7 comparing SuStaIn assignments to ground truth.

**Why this matters:** Without this experiment, a critic could argue that the null-data result simply shows SuStaIn doesn't work. This experiment proves SuStaIn works when structure exists.

## 0. Environment Setup

### 0a. Install dependencies

In [ ]:
!pip install -q pySuStaIn numpy scipy scikit-learn pandas matplotlib seaborn tqdm

### 0b. Clone repo and add to path

In [ ]:
!git clone --depth 1 https://github.com/Amelia3141/mphil.git mphil_repo 2>/dev/null || echo "Already cloned"
import sys
sys.path.insert(0, 'mphil_repo')
sys.path.insert(0, 'mphil_repo/pySuStaIn')

### 0c. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import tempfile
import warnings
from itertools import combinations
from scipy.stats import norm, mode
from scipy.linalg import cholesky
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, confusion_matrix
warnings.filterwarnings('ignore')

print("All imports successful.")

## 1. Define Constants

These are shared across all sections: symptom names, prevalences, clinical domains, correlation structure, and subtype definitions.

### 1a. Symptom names and literature-based prevalences

In [ ]:
SYMPTOM_NAMES = [
    'fatigue', 'chronic_pain', 'sleep_disturbance',           # Systemic
    'joint_pain', 'subluxations', 'joint_hypermobility',      # Musculoskeletal
    'gi_symptoms',                                             # Gastrointestinal
    'pots_dysautonomia',                                       # Autonomic
    'headaches', 'cognitive_dysfunction',                      # Neurological
    'anxiety', 'depression',                                   # Mental Health
    'mcas', 'allergies',                                       # Immune/Inflammatory
    'skin_fragility', 'urinary_symptoms', 'tmj_dysfunction',  # Other
    'vision_issues', 'hearing_tinnitus'                        # Other
]

N_BIOMARKERS = len(SYMPTOM_NAMES)
N_S_GT = np.array([3] * N_BIOMARKERS)  # 3 severity levels each

# Literature-based prevalence rates (Tinkle et al. 2017, Hakim et al. 2017)
PREVALENCES = {
    'fatigue': 0.78, 'chronic_pain': 0.82, 'sleep_disturbance': 0.65,
    'joint_pain': 0.85, 'subluxations': 0.70, 'joint_hypermobility': 0.98,
    'gi_symptoms': 0.75, 'pots_dysautonomia': 0.45,
    'headaches': 0.60, 'cognitive_dysfunction': 0.55,
    'anxiety': 0.65, 'depression': 0.45,
    'mcas': 0.25, 'allergies': 0.50,
    'skin_fragility': 0.70, 'urinary_symptoms': 0.40,
    'tmj_dysfunction': 0.35, 'vision_issues': 0.30,
    'hearing_tinnitus': 0.25
}

print(f"{N_BIOMARKERS} symptoms, {sum(N_S_GT - 1)} max events")

### 1b. Clinical domains (pre-specified, not data-derived)

In [ ]:
DOMAINS = {
    'Musculoskeletal': ['joint_pain', 'subluxations', 'joint_hypermobility'],
    'Autonomic':       ['pots_dysautonomia'],
    'Gastrointestinal':['gi_symptoms'],
    'Neurological':    ['headaches', 'cognitive_dysfunction'],
    'Mental Health':   ['anxiety', 'depression'],
    'Immune/Inflam.':  ['mcas', 'allergies'],
    'Systemic':        ['fatigue', 'chronic_pain', 'sleep_disturbance']
}

SUBTYPE_NAMES_TRUE = ['Musculoskeletal', 'Autonomic', 'Neurological']

print("7 clinical domains defined.")
for d, syms in DOMAINS.items():
    print(f"  {d}: {', '.join(syms)}")

### 1c. Correlation structure (clinical associations from literature)

In [ ]:
def build_correlation_matrix():
    """Build correlation matrix from known clinical associations."""
    n = len(SYMPTOM_NAMES)
    corr = np.eye(n)
    idx = {name: i for i, name in enumerate(SYMPTOM_NAMES)}

    def s(a, b, r):
        corr[idx[a], idx[b]] = r
        corr[idx[b], idx[a]] = r

    # Strong (r=0.45-0.60)
    s('fatigue', 'chronic_pain', 0.55)
    s('fatigue', 'sleep_disturbance', 0.50)
    s('chronic_pain', 'sleep_disturbance', 0.45)
    s('anxiety', 'depression', 0.55)
    s('joint_pain', 'subluxations', 0.50)
    s('gi_symptoms', 'mcas', 0.45)
    s('pots_dysautonomia', 'fatigue', 0.45)
    s('chronic_pain', 'depression', 0.45)

    # Moderate (r=0.25-0.40)
    s('anxiety', 'pots_dysautonomia', 0.35)
    s('fatigue', 'cognitive_dysfunction', 0.35)
    s('mcas', 'skin_fragility', 0.30)
    s('mcas', 'allergies', 0.40)
    s('headaches', 'cognitive_dysfunction', 0.35)
    s('chronic_pain', 'anxiety', 0.30)
    s('depression', 'fatigue', 0.35)
    s('pots_dysautonomia', 'cognitive_dysfunction', 0.30)
    s('joint_pain', 'joint_hypermobility', 0.35)
    s('sleep_disturbance', 'cognitive_dysfunction', 0.30)

    # Weak (r=0.15-0.25)
    s('fatigue', 'gi_symptoms', 0.20)
    s('chronic_pain', 'headaches', 0.25)
    s('anxiety', 'gi_symptoms', 0.20)
    s('skin_fragility', 'subluxations', 0.15)
    s('urinary_symptoms', 'gi_symptoms', 0.20)
    s('tmj_dysfunction', 'headaches', 0.25)
    s('vision_issues', 'headaches', 0.20)
    s('hearing_tinnitus', 'tmj_dysfunction', 0.20)

    # Force positive semi-definite
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.maximum(eigvals, 1e-6)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    np.fill_diagonal(corr, 1.0)
    return corr

CORR_MATRIX = build_correlation_matrix()
print(f"Correlation matrix: {CORR_MATRIX.shape}")
print(f"Min eigenvalue: {np.linalg.eigvalsh(CORR_MATRIX).min():.6f} (positive = valid)")

### 1d. Subtype definitions

Each subtype is defined by which symptoms have an elevated latent mean. The shift magnitudes are chosen so that domain-defining symptoms achieve Cohen's d > 1.0 between subtypes.

In [ ]:
SUBTYPE_DEFS = {
    'Musculoskeletal': {
        'proportion': 0.40,
        'elevated': {
            'joint_pain': 1.2,
            'subluxations': 1.3,
            'joint_hypermobility': 0.8,
            'chronic_pain': 0.6,       # secondary
            'skin_fragility': 0.5,
        }
    },
    'Autonomic': {
        'proportion': 0.35,
        'elevated': {
            'pots_dysautonomia': 1.4,
            'fatigue': 1.0,
            'cognitive_dysfunction': 0.8,
            'sleep_disturbance': 0.7,
            'anxiety': 0.5,            # secondary
        }
    },
    'Neurological': {
        'proportion': 0.25,
        'elevated': {
            'headaches': 1.3,
            'cognitive_dysfunction': 1.0,
            'vision_issues': 1.2,
            'tmj_dysfunction': 0.8,
            'hearing_tinnitus': 0.7,
        }
    }
}

for name, defn in SUBTYPE_DEFS.items():
    print(f"{name} ({defn['proportion']:.0%}): {', '.join(defn['elevated'].keys())}")

## 2. Generate Synthetic Data WITH Known Subtypes

The generator uses the same correlation structure and prevalences as the null-data generator, but shifts the latent means for subtype-defining symptoms. This creates genuine discrete clusters that SuStaIn should be able to recover.

### 2a. Generator function

In [ ]:
def generate_known_structure_data(n_patients=5000, seed=42):
    """
    Generate ordinal symptom data with 3 known subtypes via latent mean shifts.
    Returns DataFrame with symptom columns + subtype_true + subtype_id.
    """
    np.random.seed(seed)
    n_vars = len(SYMPTOM_NAMES)
    name_to_idx = {name: i for i, name in enumerate(SYMPTOM_NAMES)}
    L = cholesky(CORR_MATRIX, lower=True)

    subtype_labels = []
    subtype_ids = []
    all_latent = []

    subtypes = list(SUBTYPE_DEFS.keys())
    proportions = [SUBTYPE_DEFS[s]['proportion'] for s in subtypes]
    n_per_subtype = np.random.multinomial(n_patients, proportions)

    for sub_idx, (subtype_name, sub_def) in enumerate(SUBTYPE_DEFS.items()):
        n_sub = n_per_subtype[sub_idx]

        # Latent means: zero baseline + shifts for this subtype
        means = np.zeros(n_vars)
        for symptom, shift in sub_def['elevated'].items():
            means[name_to_idx[symptom]] = shift

        z = np.random.randn(n_sub, n_vars)
        latent = z @ L.T + means

        all_latent.append(latent)
        subtype_labels.extend([subtype_name] * n_sub)
        subtype_ids.extend([sub_idx] * n_sub)

    latent_all = np.vstack(all_latent)

    # Threshold to ordinal (0=absent, 1=mild, 2=severe)
    ordinal = np.zeros((n_patients, n_vars), dtype=int)
    for i, name in enumerate(SYMPTOM_NAMES):
        prev = PREVALENCES[name]
        t_absent = norm.ppf(1 - prev)
        t_severe = norm.ppf(1 - prev * 0.40)
        ordinal[:, i] = np.where(
            latent_all[:, i] < t_absent, 0,
            np.where(latent_all[:, i] < t_severe, 1, 2)
        )

    df = pd.DataFrame(ordinal, columns=SYMPTOM_NAMES)
    df['subtype_true'] = subtype_labels
    df['subtype_id'] = subtype_ids

    # Shuffle to remove ordering
    df = df.iloc[np.random.permutation(n_patients)].reset_index(drop=True)
    return df

print("Generator function defined.")

### 2b. Generate the dataset

In [ ]:
df = generate_known_structure_data(n_patients=5000, seed=42)

print(f"Generated {len(df)} patients with 3 known subtypes")
print(f"\nSubtype distribution:")
print(df['subtype_true'].value_counts().to_string())
print(f"\nPrevalence check (% with any symptoms):")
for s in SYMPTOM_NAMES[:6]:
    print(f"  {s}: {(df[s] > 0).mean():.1%} (target: {PREVALENCES[s]:.0%})")

## 3. Verify Ground Truth Effect Sizes

Before running SuStaIn, confirm the generated data actually has the intended subtype separation. Cohen's d > 1.0 should appear on domain-defining symptoms.

In [ ]:
print("Ground Truth Cohen's d Between Subtypes by Domain")
print("=" * 70)
print(f"{'Comparison':<30} {'Domain':<20} {'d':>8}  {'Sig?'}")
print("-" * 70)

for s1, s2 in combinations(df['subtype_true'].unique(), 2):
    for domain_name, symptoms in DOMAINS.items():
        d1 = df.loc[df['subtype_true'] == s1, symptoms].mean(axis=1)
        d2 = df.loc[df['subtype_true'] == s2, symptoms].mean(axis=1)
        pooled_std = np.sqrt((d1.var() + d2.var()) / 2)
        d = (d1.mean() - d2.mean()) / pooled_std if pooled_std > 0 else 0
        marker = "***" if abs(d) >= 1.0 else ("**" if abs(d) >= 0.5 else "")
        print(f"  {s1[:12]:>12} v {s2[:12]:<12}  {domain_name:<20} {d:+.3f}  {marker}")
    print()

print("*** = large (d >= 1.0), ** = medium (d >= 0.5)")

## 4. Prepare Data for Ordinal SuStaIn

In [ ]:
from pySuStaIn.OrdinalSustain import OrdinalSustain

data = df[SYMPTOM_NAMES].values.astype(float)
N_patients = data.shape[0]

output_dir = tempfile.mkdtemp(prefix='sustain_known_structure_')

print(f"Data matrix: {data.shape} (patients x symptoms)")
print(f"Severity levels per symptom: {N_S_GT[0]}")
print(f"Max possible events: {sum(N_S_GT - 1)}")
print(f"Output: {output_dir}")

## 5. Fit Ordinal SuStaIn

We fit with `N_S_max=4` (one more than truth) to test whether model selection correctly identifies k=3. Using 25 random starting points and 10,000 MCMC iterations.

**Expected runtime: 10-30 minutes.**

In [ ]:
sustain = OrdinalSustain(
    data,
    N_S_GT,
    N_BIOMARKERS,
    SYMPTOM_NAMES,
    N_startpoints=25,
    N_S_max=4,
    N_iterations_MCMC=10000,
    output_folder=output_dir,
    dataset_name='known_structure',
    use_parallel_startpoints=False,
    seed=42
)

print("SuStaIn model initialised. Fitting...")

In [ ]:
import time
t0 = time.time()
samples_sequence, samples_f, ml_subtype, ml_stage, ml_likelihood = \
    sustain.run_sustain_algorithm(plot=False)
elapsed = time.time() - t0

print(f"\nFitting complete in {elapsed/60:.1f} minutes")
print(f"Subtypes found: {len(np.unique(ml_subtype))}")
for st in sorted(np.unique(ml_subtype)):
    n = np.sum(ml_subtype == st)
    print(f"  Subtype {int(st)}: {n} patients ({n/len(ml_subtype):.1%})")

## 6. Cross-Validation for Model Selection

10-fold cross-validation using CVIC (Cross-Validation Information Criterion). The optimal k should be 3 (matching the true number of subtypes).

In [ ]:
print("Running 10-fold cross-validation...")
t0 = time.time()

try:
    CVIC, loglike = sustain.cross_validate_sustain_model(test_idxs=None, select_fold=None)
    cv_elapsed = time.time() - t0
    optimal_k = np.argmin(CVIC) + 1

    print(f"\nCompleted in {cv_elapsed/60:.1f} minutes")
    print(f"\nCVIC values:")
    for k in range(len(CVIC)):
        marker = " <-- optimal" if k + 1 == optimal_k else ""
        print(f"  k={k+1}: CVIC = {CVIC[k]:.2f}{marker}")

    print(f"\nOptimal k: {optimal_k} (true: 3)")
    print(f"Model selection {'CORRECT' if optimal_k == 3 else 'INCORRECT'}")

except Exception as e:
    print(f"Cross-validation failed: {e}")
    print("Proceeding with the number of subtypes from the main fit.")
    optimal_k = len(np.unique(ml_subtype))

## 7. Assess Recovery via Adjusted Rand Index

ARI measures agreement between SuStaIn assignments and ground truth, adjusted for chance. It handles label permutation automatically (subtype 0 in SuStaIn might correspond to subtype 2 in ground truth).

### 7a. Compute ARI

In [ ]:
ground_truth = df['subtype_id'].values
ari = adjusted_rand_score(ground_truth, ml_subtype)

print("=" * 60)
print("KNOWN-STRUCTURE RECOVERY")
print("=" * 60)
print(f"Adjusted Rand Index:  {ari:.3f}")
print(f"Success threshold:    ARI > 0.7")
print(f"Result:               {'PASS' if ari > 0.7 else 'FAIL'}")
print("=" * 60)

### 7b. Find best label mapping (Hungarian algorithm)

In [ ]:
n_found = len(np.unique(ml_subtype))
n_true = len(np.unique(ground_truth))
cost = np.zeros((n_found, n_true))

for i in range(n_found):
    for j in range(n_true):
        cost[i, j] = -np.sum((ml_subtype == i) & (ground_truth == j))

row_ind, col_ind = linear_sum_assignment(cost)
mapping = dict(zip(row_ind, col_ind))

print("Best label mapping (SuStaIn -> Ground Truth):")
for r, c in zip(row_ind, col_ind):
    matched = np.sum((ml_subtype == r) & (ground_truth == c))
    total = np.sum(ml_subtype == r)
    print(f"  SuStaIn {r} -> {SUBTYPE_NAMES_TRUE[c]}  ({matched}/{total} = {matched/total:.1%})")

### 7c. Remapped accuracy

In [ ]:
remapped = np.zeros_like(ml_subtype)
for old, new in mapping.items():
    remapped[ml_subtype == old] = new
accuracy = np.mean(remapped == ground_truth)

print(f"Remapped classification accuracy: {accuracy:.1%}")
print(f"(Chance level for 3 subtypes: ~33%)")

## 8. Evaluate Against Pre-Specified Validation Criteria

These are the same 6 criteria used in the null-data test. For known-structure data, criteria 1, 2, and 5 should PASS (confirming the framework accepts genuine subtypes).

### 8a. Criterion 1: Prevalence (>= 10%)

In [ ]:
min_prev = min(
    np.sum(ml_subtype == s) / len(ml_subtype)
    for s in np.unique(ml_subtype)
)
c1_pass = min_prev >= 0.10

print(f"Criterion 1 (Prevalence >= 10%): {'PASS' if c1_pass else 'FAIL'}")
print(f"  Minimum subtype prevalence: {min_prev:.1%}")
for s in sorted(np.unique(ml_subtype)):
    n = np.sum(ml_subtype == s)
    print(f"  Subtype {int(s)}: {n/len(ml_subtype):.1%}")

### 8b. Criterion 2: Distinctiveness (Cohen's d >= 0.5 on >= 3 domains)

In [ ]:
n_domains_passing = 0
print("Criterion 2 (Distinctiveness):")
print(f"  {'Domain':<25} {'Max d':>8}  {'Pass?'}")
print(f"  {'-'*45}")

for domain_name, symptoms in DOMAINS.items():
    max_d = 0
    for s1, s2 in combinations(np.unique(ml_subtype), 2):
        d1 = df.loc[ml_subtype == s1, symptoms].mean(axis=1)
        d2 = df.loc[ml_subtype == s2, symptoms].mean(axis=1)
        pooled = np.sqrt((d1.var() + d2.var()) / 2)
        d = abs(d1.mean() - d2.mean()) / pooled if pooled > 0 else 0
        max_d = max(max_d, d)
    passes = max_d >= 0.5
    if passes:
        n_domains_passing += 1
    print(f"  {domain_name:<25} {max_d:8.3f}  {'PASS' if passes else 'FAIL'}")

c2_pass = n_domains_passing >= 3
print(f"\n  Domains passing: {n_domains_passing}/7")
print(f"  Criterion 2: {'PASS' if c2_pass else 'FAIL'} (need >= 3)")

### 8c. Criterion 5: Not severity-only (r < 0.5)

In [ ]:
total_burden = df[SYMPTOM_NAMES].sum(axis=1)
r_severity = np.corrcoef(ml_subtype, total_burden)[0, 1]
c5_pass = abs(r_severity) < 0.5

print(f"Criterion 5 (Not severity-only):")
print(f"  Correlation with total burden: r = {r_severity:.3f}")
print(f"  Threshold: |r| < 0.5")
print(f"  Result: {'PASS' if c5_pass else 'FAIL'}")

### 8d. Summary table

In [ ]:
print("=" * 70)
print("PRE-SPECIFIED CRITERIA SUMMARY (KNOWN-STRUCTURE DATA)")
print("=" * 70)
print(f"{'Criterion':<45} {'Result':<8} {'Value'}")
print("-" * 70)
print(f"{'1. Prevalence >= 10%':<45} {'PASS' if c1_pass else 'FAIL':<8} min {min_prev:.1%}")
print(f"{'2. Distinctiveness (d>=0.5 on >=3 domains)':<45} {'PASS' if c2_pass else 'FAIL':<8} {n_domains_passing}/7 domains")
print(f"{'3. Bootstrap stability (ARI >= 0.6)':<45} {'N/A':<8} (separate experiment)")
print(f"{'4. Model selection robustness':<45} {'--':<8} optimal k = {optimal_k}")
print(f"{'5. Not severity-only (|r| < 0.5)':<45} {'PASS' if c5_pass else 'FAIL':<8} r = {r_severity:.3f}")
print(f"{'6. Plausible sequences':<45} {'--':<8} (requires clinical review)")
print("-" * 70)
print(f"{'Recovery ARI':<45} {'PASS' if ari > 0.7 else 'FAIL':<8} ARI = {ari:.3f}")
print(f"{'Remapped accuracy':<45} {'--':<8} {accuracy:.1%}")
print("=" * 70)

print("\nContrast with null-data test:")
print("  Null data:  Crit 1 PASS, Crit 2 FAIL (max d=0.27), Crit 5 PASS")
print(f"  Known str:  Crit 1 {'PASS' if c1_pass else 'FAIL'}, Crit 2 {'PASS' if c2_pass else 'FAIL'} ({n_domains_passing} domains), Crit 5 {'PASS' if c5_pass else 'FAIL'}")
print("\nThis confirms the framework discriminates: rejects artefacts, accepts real subtypes.")

## 9. Visualise Results

### 9a. Subtype profiles: ground truth vs SuStaIn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

# Ground truth
for i, st_name in enumerate(SUBTYPE_NAMES_TRUE):
    mask = df['subtype_true'] == st_name
    means = df.loc[mask, SYMPTOM_NAMES].mean()
    axes[0].barh(range(N_BIOMARKERS), means, alpha=0.6, label=st_name)
axes[0].set_yticks(range(N_BIOMARKERS))
axes[0].set_yticklabels(SYMPTOM_NAMES, fontsize=9)
axes[0].set_xlabel('Mean Severity (0-2)')
axes[0].set_title('Ground Truth Subtype Profiles')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].invert_yaxis()

# SuStaIn-recovered
for s in sorted(np.unique(ml_subtype)):
    mask = ml_subtype == s
    means = df.loc[mask, SYMPTOM_NAMES].mean()
    label = SUBTYPE_NAMES_TRUE[mapping[int(s)]] if int(s) in mapping else f'Subtype {int(s)}'
    axes[1].barh(range(N_BIOMARKERS), means, alpha=0.6, label=f'SuStaIn {int(s)} ({label})')
axes[1].set_xlabel('Mean Severity (0-2)')
axes[1].set_title('SuStaIn-Recovered Subtype Profiles')
axes[1].legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('known_structure_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

### 9b. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(ground_truth, remapped)
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=SUBTYPE_NAMES_TRUE,
            yticklabels=SUBTYPE_NAMES_TRUE)
ax.set_xlabel('Predicted Subtype')
ax.set_ylabel('True Subtype')
ax.set_title(f'Recovery Confusion Matrix\nARI = {ari:.3f}, Accuracy = {accuracy:.1%}')
plt.tight_layout()
plt.savefig('known_structure_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Results

In [ ]:
results = {
    'ari': float(ari),
    'remapped_accuracy': float(accuracy),
    'optimal_k_cvic': int(optimal_k),
    'true_k': 3,
    'n_subtypes_found': int(len(np.unique(ml_subtype))),
    'criterion_1_prevalence_min': float(min_prev),
    'criterion_1_pass': bool(c1_pass),
    'criterion_2_domains_passing': int(n_domains_passing),
    'criterion_2_pass': bool(c2_pass),
    'criterion_5_severity_r': float(r_severity),
    'criterion_5_pass': bool(c5_pass),
}

with open('known_structure_recovery_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to known_structure_recovery_results.json")
print(json.dumps(results, indent=2))